# 10.P · Solution: deploy the churn model and the chatbot

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/10-deployment-mlops/project/solution.ipynb)

The brief is in the [README](README.md). This notebook runs every part of the project the way an automated check would: it trains and records the model, exercises the API, writes a monitoring report from the API's prediction log, tests the chatbot, and smoke-tests the app. To use the programs yourself, see "Running it" in the README.

In [1]:
# --- course files: found locally, downloaded on Colab -------------------------
import sys, urllib.request
from pathlib import Path
REPO_RAW = "https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/"

def repo_file(rel):
    """Return a local path to a file in this repo, downloading it if needed."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / rel).exists():
            return parent / rel
    dest = Path("course_files") / rel
    dest.parent.mkdir(parents=True, exist_ok=True)
    if not dest.exists():
        urllib.request.urlretrieve(REPO_RAW + rel, dest)
    return dest

import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
repo_file("data/download.py")                                  # churn_model finds the data through it
sys.path.insert(0, str(repo_file("lib/churn_model.py").parent))
import churn_model as cm

bundle = cm.load_bundle()          # trains and saves models/churn_bundle.joblib the first time
ROOT = cm.repo_root()
print(f"Model trained {bundle['trained']}, test AUC {bundle['metrics']['test_auc']}, "
      f"threshold {bundle['threshold']}")

import importlib.util

def load_module(rel, name):
    """Import a .py file from the course as a module (so the notebook can test it)."""
    spec = importlib.util.spec_from_file_location(name, repo_file(rel))
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

import os
sys.path.insert(0, str(repo_file("modules/10-deployment-mlops/project/support_bot.py").parent))
PROJECT = "modules/10-deployment-mlops/project/"

Model trained 2026-10-28, test AUC 0.741, threshold 0.07


## 1. Train and record (`train.py`)

One function call trains the model, saves the bundle, and logs the run (parameters, metrics and the bundle file) to MLflow. The bundle now carries its MLflow run ID, so the file and the record point at each other.

In [2]:
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
train = load_module(PROJECT + "train.py", "train")
run_id = train.main()
bundle = cm.load_bundle()
print("bundle knows its run:", bundle.get("mlflow_run_id") == run_id)

Trained and saved churn_bundle.joblib: test AUC 0.741, threshold 0.07, MLflow run d8d295b3
bundle knows its run: True


## 2. The API (`api.py`)

In [3]:
log_path = ROOT / "models" / "prediction_log.csv"
os.environ["PREDICTION_LOG"] = str(log_path)
if log_path.exists():
    log_path.unlink()                    # a fresh log for this exercise

from fastapi.testclient import TestClient
api = load_module(PROJECT + "api.py", "project_api")
client = TestClient(api.app)
print(client.get("/health").json())
print(client.get("/model").json()["purpose"])

{'status': 'ok', 'model_trained': '2026-10-28', 'test_auc': 0.741, 'threshold': 0.07, 'mlflow_run_id': 'd8d295b3bc2f4781a5fdf61db2ccb383'}
Rank customers by churn risk to target win-back vouchers. Not for any other decision.


In [4]:
customer = {"orders": 2, "spend": 310, "products": 14, "tenure_days": 150, "days_since_last": 95}
print(client.post("/predict", json=customer).json())
print(client.post("/predict", json={**customer, "spend": -1}).status_code, "for a negative spend")

{'churn_probability': 0.5244, 'voucher': True, 'threshold': 0.07}
422 for a negative spend


## 3. A live month

It is 1 December 2011. Score every customer as production would: features from the 274 days before today (the same window as training), sent through the API in one batch. Every prediction lands in the log.

In [5]:
live = cm.build_customer_table(cm.load_sales(), "2011-12-01", lookback_days=274)
log_path.unlink(missing_ok=True)       # start the month with an empty log (the test calls above were logged too)
payload = live[cm.FEATURES].astype({"spend": float}).to_dict(orient="records")
response = client.post("/predict/batch", json=payload)
log = pd.read_csv(log_path)
print(response.status_code, f"{len(log):,} predictions logged; vouchers for {log['voucher'].mean():.0%}")
log.tail(3)

200 3,956 predictions logged; vouchers for 92%


,timestamp,model_trained,orders,spend,products,tenure_days,days_since_last,churn_probability,voucher
3953,2026-10-28T15:59:01,2026-10-28,1,100.21,7,117,117,0.6306,True
3954,2026-10-28T15:59:01,2026-10-28,12,1522.83,201,223,0,0.0206,False
3955,2026-10-28T15:59:01,2026-10-28,3,1837.28,59,192,33,0.2999,True


## 4. Monitoring report

Compare the logged inputs and scores with the training customers (10.6). Labels for this month arrive on 1 March 2012, so the report can only judge drift today.

In [6]:
from sklearn.model_selection import train_test_split
cust = cm.build_customer_table(cm.load_sales())
train_rows, _ = train_test_split(cust, test_size=0.25, random_state=42, stratify=cust["churned"])
p_train = bundle["pipeline"].predict_proba(train_rows[cm.FEATURES])[:, 1]

def band(v):
    return "act" if v > 0.25 else "watch" if v > 0.1 else "ok"

report = pd.DataFrame({f: {"PSI": cm.psi(train_rows[f], log[f]), "training median": train_rows[f].median(),
                           "this month median": log[f].median()} for f in cm.FEATURES}).T
report.loc["churn score"] = [cm.psi(p_train, log["churn_probability"]), p_train.mean(), log["churn_probability"].mean()]
report["status"] = report["PSI"].map(band)
report.round(3)

,PSI,training median,this month median,status
orders,0.007,2.000,2.000,ok
spend,0.010,539.130,622.340,ok
products,0.017,28.000,33.000,ok
tenure_days,0.269,184.000,182.500,act
days_since_last,0.151,72.000,41.000,watch
churn score,0.069,0.436,0.403,ok


**Reading the report.**

- `tenure_days` says **act**, but 10.6 showed why: its distribution depends on where the 274-day window starts. Training's window began on 1 December 2010, in the pre-Christmas rush; today's began in early March. Fix the feature (for example, measure tenure over all history) rather than reacting to the alarm.
- `days_since_last` says **watch**: on 1 December most customers have just ordered for Christmas (median 41 days, against 72 at training).
- The **score** distribution is stable: the model predicts an average churn of 40%.

**But 10.6 showed that December is exactly when calibration breaks:** a year earlier the model predicted 39% and 65% churned. The honest recommendation is: *the inputs look like a normal December; expect actual churn well above the predicted 40%; if vouchers are cheap, the campaign is still worth sending (the threshold is far below either number); check the labels on 1 March 2012 and compare with last December before retraining.*

## 5. The chatbot (`support_bot.py`)

> **About the chatbot outputs.** They were produced with `llama3.2` (3B) via Ollama. Yours will differ with another model or provider, and may differ between runs.

In [7]:
import support_bot, time
print(support_bot.llm_status())
start = time.time()
index = support_bot.build_index()
print(f"{len(index[1])} chunks from {len({c['source'] for c in index[1]})} documents in {time.time() - start:.1f} s")

(True, 'ollama · llama3.2')


26 chunks from 11 documents in 4.0 s


In [8]:
questions = ["How long do I have to return a sale item?",
             "Do you deliver to Norway, and how much does it cost?",
             "Can I pay with Bitcoin?"]
for q in questions:
    start = time.time()
    reply, hits = support_bot.answer(index, q)
    print(f"Q: {q}\nA: {reply}\n   sources searched: {[h['source'] for h in hits]} ({time.time() - start:.1f} s)\n")

Q: How long do I have to return a sale item?
A: You have 14 days to return a sale item. [returns.md]
   sources searched: ['returns.md', 'christmas.md', 'returns.md'] (0.3 s)



Q: Do you deliver to Norway, and how much does it cost?
A: I don't know.
   sources searched: ['international-shipping.md', 'international-shipping.md', 'wholesale.md'] (0.3 s)



Q: Can I pay with Bitcoin?
A: I don't know.
   sources searched: ['wholesale.md', 'refunds.md', 'damaged-items.md'] (0.2 s)



- The returns answer is right and cited.
- Bitcoin is not in the documents, and the bot says so instead of inventing a policy.
- **Norway is a miss.** No document names Norway, but `international-shipping.md` covers it: outside the EU, "rest of the world", £14.95, 7 to 14 working days. A 3B model will not make that step and plays safe with "I don't know". A larger model usually would. This is the kind of failure an evaluation set (09.11) should include, and one a better document (a list of countries) fixes more reliably than a bigger model.

## 6. Smoke-test the app (`app.py`)

In [9]:
from streamlit.testing.v1 import AppTest
from streamlit import logger as st_logger
st_logger.set_log_level("error")              # hide "missing ScriptRunContext": harmless outside a browser
import logging

def new_app(path, timeout=60):
    """A fresh AppTest, with Streamlit's harmless notices about running outside a browser hidden."""
    for name in list(logging.root.manager.loggerDict):
        if name.startswith("streamlit"):
            logging.getLogger(name).setLevel(logging.ERROR)
    return AppTest.from_file(path, default_timeout=timeout)

at = new_app(str(repo_file(PROJECT + "app.py")), timeout=180).run()
print("exceptions:", list(at.exception))
print("tabs:", [t.label for t in at.tabs])
print("churn tab:", at.metric[0].value, "|", at.success[0].value if len(at.success) else at.info[0].value)

exceptions: []
tabs: ['Churn risk', 'Support chatbot']
churn tab: 52% | Send a £5 voucher (break-even risk 8%).


In [10]:
at.chat_input[0].set_value("Is gift wrapping free?").run()
for message in at.chat_message:
    print(f"{message.name}: {message.markdown[0].value}")

user: Is gift wrapping free?
assistant: No, gift wrapping is not free. The cost of gift wrapping is **£3.50 per item** [gift-wrap.md].

*Searched: gift-wrap.md, vouchers-and-promo-codes.md*


And without an LLM: point the app at a provider with no key, and the chatbot tab explains what to do instead of crashing.

In [11]:
saved = {k: os.environ.get(k) for k in ["LLM_PROVIDER", "OPENAI_API_KEY"]}
os.environ["LLM_PROVIDER"], os.environ["OPENAI_API_KEY"] = "openai", ""
import course_llm; course_llm.PROVIDER = None                  # forget the earlier choice
print(support_bot.llm_status())
at_no_llm = new_app(str(repo_file(PROJECT + "app.py")), timeout=180).run()
print("exceptions:", list(at_no_llm.exception), "| warning shown:", at_no_llm.warning[0].value.split(".")[0])
for k, v in saved.items():
    if v is None: os.environ.pop(k, None)
    else: os.environ[k] = v
course_llm.PROVIDER = None

(False, 'LLM_PROVIDER=openai needs OPENAI_API_KEY in .env')
exceptions: [] | warning shown: The chatbot needs a language model, and none is configured


`llm_status()` checks that a provider is chosen and has what it needs (a key, or a running Ollama). It cannot tell whether a key is *valid* without spending a request: a wrong key is reported on the first question instead.

## What the solution found

- The production bundle reproduces 10.1 (test AUC 0.741, threshold 0.07) and records its MLflow run ID.
- The API logged 3,956 predictions for 1 December 2011. Inputs: tenure "act" (a window artefact, 10.6), recency "watch" (Christmas shopping); scores stable.
- The real risk this month is calibration, which no input alarm shows: based on last December, expect far more churn than the 40% predicted.
- The chatbot answered the returns question correctly with a citation, refused the Bitcoin question, and missed the Norway question that the documents cover only indirectly.
- The app runs with and without an LLM.